# Dataset Analysis
Inspect UCF-Crime, DCSASS, and ShanghaiTech before training. Missing roots and unknown labels are reported, not guessed.

In [ ]:
import sys
import platform
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from IPython.display import display

print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'PyTorch: {torch.__version__}')
print(f'TorchVision: {torchvision.__version__}')
print(f'OpenCV: {cv2.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'CUDA device: {torch.cuda.get_device_name(0)}')

In [ ]:
import os

ROOT = Path.cwd()
CANDIDATES = {
    "UCF-Crime": [ROOT / "UCF-Crime", ROOT / "UCF_Crime", ROOT / "UCF-Crime Dataset"],
    "DCSASS": [ROOT / "DCSASS Dataset" / "DCSASS Dataset", ROOT / "DCSASS Dataset"],
    "ShanghaiTech": [ROOT / "SHANGHAI", ROOT / "ShanghaiTech", ROOT / "ShanghaiTech Campus"],
}
SEARCH_ROOTS = [
    ROOT.parent,
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
    Path("D:/datasets"),
    Path("D:/data"),
]

def find_named_dataset(names, max_depth=3):
    normalized_names = {name.lower() for name in names}
    found = []
    for search_root in SEARCH_ROOTS:
        if not search_root.is_dir():
            continue
        for current_root, directories, _ in os.walk(search_root):
            current_path = Path(current_root)
            depth = len(current_path.relative_to(search_root).parts)
            if depth >= max_depth:
                directories[:] = []
            for directory in directories:
                if directory.lower() in normalized_names:
                    found.append(current_path / directory)
    return sorted(set(found))

ucf_names = ["UCF-Crime", "UCF_Crime", "UCF-Crime Dataset", "UCF Crime"]
CANDIDATES["UCF-Crime"].extend(find_named_dataset(ucf_names))

def resolve(candidates):
    return next((path for path in candidates if path.is_dir()), None)

DATASET_ROOTS = {name: resolve(paths) for name, paths in CANDIDATES.items()}
for name, path in DATASET_ROOTS.items():
    if path:
        print(f"{name}: FOUND at {path}")
    else:
        print(f"{name}: NOT FOUND in workspace or reasonable search locations")
        if name == "UCF-Crime":
            print("Set DATASET_ROOTS['UCF-Crime'] to the actual folder path before continuing.")


In [ ]:
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.mpeg', '.mpg'}
FRAME_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
def print_structure(root, depth=3, limit=80):
    if root is None: return
    shown = 0
    for path in sorted(root.rglob('*'), key=lambda item: str(item).lower()):
        relative = path.relative_to(root)
        if len(relative.parts) <= depth:
            print('  ' * (len(relative.parts) - 1) + ('[D] ' if path.is_dir() else '[F] ') + relative.name)
            shown += 1
            if shown >= limit: print('... truncated ...'); break
for name, root in DATASET_ROOTS.items():
    print(f'\n===== {name} =====')
    print_structure(root)

In [ ]:
def discover(root, extensions):
    if root is None: return []
    return sorted(path for path in root.rglob('*') if path.is_file() and path.suffix.lower() in extensions)
def class_name(path, root):
    ignored = {'frames', 'frame', 'label', 'labels', 'train', 'test', 'training', 'testing'}
    for part in reversed(path.relative_to(root).parts[:-1]):
        if part.lower() not in ignored: return part
    return 'Unknown'
def infer_label(dataset, path, root):
    text = ' '.join(part.lower() for part in path.relative_to(root).parts)
    if any(token in text for token in ('normal', 'normalvideos', 'normal_video')): return 0, 'Normal'
    if dataset == 'ShanghaiTech': return None, 'Needs label-file mapping'
    if dataset in {'UCF-Crime', 'DCSASS'}: return 1, 'Anomaly'
    return None, 'Unknown'
discovered = {}
for name, root in DATASET_ROOTS.items():
    videos, frames = discover(root, VIDEO_EXTENSIONS), discover(root, FRAME_EXTENSIONS)
    discovered[name] = {'videos': videos, 'frames': frames}
    print(f'{name}: {len(videos):,} videos; {len(frames):,} frames')

In [ ]:
records = []
for dataset, root in DATASET_ROOTS.items():
    if root is None: continue
    for source_type in ('videos', 'frames'):
        for path in discovered[dataset][source_type]:
            label, inferred_class = infer_label(dataset, path, root)
            records.append({'video_path': str(path.resolve()), 'dataset_name': dataset, 'label': label, 'class_name': inferred_class, 'source_type': source_type, 'folder_class': class_name(path, root)})
metadata = pd.DataFrame(records)
if metadata.empty: print('No supported files found.')
else:
    display(metadata.head(10))
    display(metadata.groupby(['dataset_name', 'source_type', 'class_name'], dropna=False).size().rename('count').reset_index())
    print('Rows needing label review:')
    display(metadata[metadata.label.isna()].head(20))

In [ ]:
def validate_video(path):
    capture = cv2.VideoCapture(str(path))
    readable = capture.isOpened() and capture.get(cv2.CAP_PROP_FRAME_COUNT) > 0
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT)) if readable else 0
    fps = float(capture.get(cv2.CAP_PROP_FPS)) if readable else 0.0
    capture.release()
    return readable, frame_count, fps
checks = []
for dataset, values in discovered.items():
    for path in values['videos']:
        readable, frames, fps = validate_video(path)
        checks.append({'dataset_name': dataset, 'video_path': str(path), 'readable': readable, 'frame_count': frames, 'fps': fps})
video_checks = pd.DataFrame(checks)
if video_checks.empty: print('No videos available for OpenCV validation.')
else:
    display(video_checks.groupby(['dataset_name', 'readable']).size().rename('count').reset_index())
    display(video_checks.head(10))

In [ ]:
import matplotlib.pyplot as plt
def show_samples(paths, title, limit=6):
    paths = [path for path in paths if path.suffix.lower() in FRAME_EXTENSIONS][:limit]
    if not paths: print(f'No frame samples for {title}'); return
    figure, axes = plt.subplots(1, len(paths), figsize=(3 * len(paths), 3)); axes = np.atleast_1d(axes)
    for axis, path in zip(axes, paths):
        try:
            with Image.open(path) as image: axis.imshow(image.convert('RGB'))
            axis.set_title(path.parent.name); axis.axis('off')
        except Exception as error: axis.set_title(type(error).__name__); axis.axis('off')
    figure.suptitle(title); plt.tight_layout(); plt.show()
for dataset, values in discovered.items(): show_samples(values['frames'], f'{dataset} sample frames')

In [2]:
OUTPUT_DIR = ROOT / 'metadata'; OUTPUT_DIR.mkdir(exist_ok=True)
if metadata.empty: print('Metadata CSV not written because no files were found.')
else:
    output_path = OUTPUT_DIR / 'unified_metadata.csv'
    metadata.to_csv(output_path, index=False)
    print(f'Saved unified metadata: {output_path}')
    print('Review missing labels and add the actual UCF-Crime path before dataset preparation.')

NameError: name 'ROOT' is not defined